# Module 08 — Files, CSV and JSON

From here on the data is not written in the source. `data/sensors.csv` sits next to
this notebook and stays with the course to the end: tested in module 15, fetched
over HTTP in 16, analysed in 18, stored in SQLite in 19.

Every file below is opened in a `with` block. What `with` is — the protocol, and how
to write your own — is module 09. Here it is enough that the file gets closed when
the block ends, including when it ends because something raised.

## 1. A path is an object

`pathlib.Path` is the type you have been reading in every test file since module 00.
It is not a string: it knows what a separator is, so one expression works on Windows
and on macOS.

In [ ]:
from pathlib import Path

# A notebook has no __file__, so it has to work out where it is from the directory
# the kernel started in -- which is this folder if you opened the notebook here, and
# the top of the repository if you started Jupyter there.
HERE = Path.cwd() if (Path.cwd() / "data").is_dir() else Path.cwd() / "08_files"
DATA = HERE / "data"

sensors = DATA / "sensors.csv"  # `/` joins parts -- no separator to get wrong

print(sensors.name)  # sensors.csv
print(sensors.stem)  # sensors
print(sensors.suffix)  # .csv
print(sensors.parent.name)
print(sensors.exists())

Java has `File` and, since Java 7, `Path` and `Files` — the same idea, with the
methods on a separate class. In Python they are on the object: `data.read_text()`,
not `Files.readString(data)`.

`Path.cwd()` is the directory the program was started in, which is a poor thing to
depend on. The reliable way to find a file that ships with the code is to start from
the code:

```python
HERE = Path(__file__).resolve().parent   # the folder this .py file is in
DATA = HERE / "data" / "sensors.csv"
```

`__file__` does not exist in a notebook, which is why the cell above uses `cwd()`.
The exercises use the `__file__` form, and so should anything you ship.

## 2. Reading, and the argument you must not leave out

For a file that fits in memory, `read_text` is the whole operation. The second
argument is the one that matters.

In [ ]:
sensors = DATA / "sensors.csv"
text = sensors.read_text(encoding="utf-8")

print(text)
print(len(text), "code points |", len(sensors.read_bytes()), "bytes")

Five more bytes than code points, and five `°` in the file — module 07's arithmetic,
now on something that came off a disk.

**Leave `encoding=` out and Python asks the operating system.** The answer is UTF-8
on most Linux and macOS systems and has historically been something else on Windows,
which is the mechanism behind a whole genre of "works on my machine". Python 3.15
changes the default to UTF-8; until every machine you care about runs it, write the
argument.

You do not have to take that on trust. Python will find the lines for you:

```
uv run python -X warn_default_encoding your_script.py
```

```
your_script.py:4: EncodingWarning: 'encoding' argument not specified
  p.write_text("hallo\n")
```

## 3. The wrong encoding, twice

Two ways to get it wrong, and the loud one is the lucky one.

In [ ]:
sensors = DATA / "sensors.csv"

try:
    sensors.read_text(encoding="ascii")
except UnicodeDecodeError as err:
    print("ascii   ->", err)

In [ ]:
print("latin-1 -> no exception at all:")
print((DATA / "sensors.csv").read_text(encoding="latin-1").splitlines()[1])

`latin-1` maps every one of the 256 possible bytes to a character, so there is no
input it can fail to decode. What comes out is `Â°C`: the two bytes UTF-8 spends on
`°` read as two separate characters. (Writing is the other direction, and it does
raise — there are 256 characters on offer, so `"€".encode("latin-1")` has nowhere to
put the euro sign.)

That string is now a perfectly ordinary `str`. It goes into your dict, your database,
your report. Nothing raises, and the damage is found weeks later by a human reading
output. **An exception is the good outcome here** — the failure mode worth fearing is
the one that returns a value.

## 4. Line by line

`read_text` loads the whole file. For a log that does not fit in memory, iterate the
file object instead: it yields one line at a time and reads as it goes.

In [ ]:
with open(DATA / "sensors.csv", encoding="utf-8") as fh:
    for number, line in enumerate(fh, start=1):
        print(number, repr(line))
        if number == 3:
            break

Note the `\n` at the end of each one: iterating a file keeps the line ending, because
it is handing you what is in the file. `.rstrip("\n")` or `.strip()` removes it.

`splitlines()` on the whole text does not have that problem — it splits *at* the line
endings and keeps none of them. It also gets `\r\n` right, which `split("\n")` does
not.

In [ ]:
text = (DATA / "sensors.csv").read_text(encoding="utf-8")

print(text.splitlines()[:2])

with_crlf = "a\r\nb\r\n"
print(with_crlf.splitlines())  # ['a', 'b']
print(with_crlf.split("\n"))  # ['a\r', 'b\r', ''] -- the \r is still there

## 5. Writing

The mode is the second argument to `open`. Three are worth knowing.

| mode | what it does |
| --- | --- |
| `"w"` | write — **truncates the file to nothing the moment it opens**, before you write a byte |
| `"a"` | append — writes at the end, creates the file if it is not there |
| `"x"` | create — raises `FileExistsError` rather than touch an existing file |

`"w"` on the wrong path destroys the file. `"x"` is the one to reach for when you
mean "this should be new".

In [ ]:
import tempfile

work = Path(tempfile.mkdtemp())  # a directory that is thrown away with the system's temp files
target = work / "report.txt"

with open(target, "w", encoding="utf-8") as fh:
    fh.write("TH-04;91.0\n")  # write() takes a string and adds no newline of its own

with open(target, "a", encoding="utf-8") as fh:
    fh.write("TH-09;23.1\n")

print(repr(target.read_text(encoding="utf-8")))

try:
    with open(target, "x", encoding="utf-8") as fh:
        fh.write("this would replace the file")
except FileExistsError as err:
    print(type(err).__name__, "-- x refuses to overwrite")

`Path.write_text(text, encoding="utf-8")` is the short form of the `"w"` case,
truncation included.

## 6. CSV

Splitting on the separator works until a field contains one. Then it stops working
silently, on somebody else's data, in production. The `csv` module knows about
quoting.

In [ ]:
import csv

with open(DATA / "sensors.csv", newline="", encoding="utf-8") as fh:
    for row in csv.reader(fh, delimiter=";"):
        print(row)

`csv.DictReader` uses the first row as the field names and hands back a dict per
line — module 06, arriving where it is actually used.

In [ ]:
import csv

with open(DATA / "sensors.csv", newline="", encoding="utf-8") as fh:
    rows = list(csv.DictReader(fh, delimiter=";"))

print(rows[0])

# Every value the reader produces has the same type. Which one?
assert type(rows[0]["value"]) is ...

A CSV file has no types — it is text. Converting is your job, and it is the step where
bad data announces itself:

```python
value = float(row["value"])   # ValueError on anything that is not a number
```

**`newline=""`** is in every one of these calls and is not decoration. The `csv`
module handles line endings itself, because a quoted field may contain one. Without
`newline=""` the text layer translates line endings as well, and on Windows a written
file gets `\r\r\n`. The documentation says to pass it; here is the reason, visible on
any platform:

In [ ]:
import csv
import tempfile

target = Path(tempfile.mkdtemp()) / "out.csv"

with open(target, "w", newline="", encoding="utf-8") as fh:
    writer = csv.writer(fh, delimiter=";")
    writer.writerow(["tag", "value"])
    writer.writerow(["TH;09", 23.1])  # a field containing the separator

print(target.read_bytes())  # \r\n -- written by the csv module, not by the file object
print(target.read_text(encoding="utf-8"))

The field with the separator in it came out quoted, and a reader gives it back as one
field. That is the whole argument for the module.

## 7. JSON

`json.loads` takes a string, `json.load` takes an open file; `dumps` and `dump` go
the other way. The mapping to Python types is direct — and lossy in two places, with a third that
refuses outright.

In [ ]:
import json

limits = json.loads((DATA / "limits.json").read_text(encoding="utf-8"))

print(limits)
print(limits["sensors"]["TH-04"]["high"], type(limits["strict"]), limits.get("missing"))

| JSON | Python |
| --- | --- |
| object | `dict` |
| array | `list` |
| string | `str` |
| number | `int` or `float` |
| `true` / `false` | `True` / `False` |
| `null` | `None` |

Now the round trip. Predict all three.

In [ ]:
import json

original = {"tags": ("TH-04", "TH-09"), 3: "an int key"}
back = json.loads(json.dumps(original))

assert type(back["tags"]) is ...
assert list(back) == ...  # what happened to the key 3?

JSON has arrays but no tuples, and its object keys are strings by definition — so a
tuple comes back a list and an `int` key comes back a `str` key, both without a word.
A `set` is refused outright, which at least happens at the line that wrote it.

In [ ]:
import json

try:
    json.dumps({"seen": {"TH-04", "TH-09"}})
except TypeError as err:
    print("TypeError:", err)

print(json.dumps({"unit": "°C"}))  # escaped by default
print(json.dumps({"unit": "°C"}, ensure_ascii=False))  # readable
print(json.dumps({"b": 1, "a": 2}, indent=2, sort_keys=True))

`ensure_ascii=True` is the default and makes the output pure ASCII by escaping
everything else. It is safe and unreadable; when the file is for a person, turn it
off and name the encoding when you write.

## 8. Bytes

Under all of this, a file is bytes. Opening in binary mode says so.

In [ ]:
raw = (DATA / "sensors.csv").read_bytes()
print(type(raw), len(raw))
print(raw[:22])

text = raw.decode("utf-8")  # exactly what read_text does after reading
print(type(text), len(text))

`read_text` is `read_bytes` plus a `decode`, and the encoding is which decode. That is
the whole story here; module 16 is where it gets interesting, because HTTP hands you
bytes with the encoding declared in a header that is sometimes wrong.

## 9. When the file is not there

`FileNotFoundError`, and it is the right behaviour: a program that carries on with an
empty string because the config was missing is harder to debug than one that stops.

In [ ]:
missing = DATA / "nope.csv"

print(missing.exists())

try:
    missing.read_text(encoding="utf-8")
except FileNotFoundError as err:
    print(type(err).__name__, "-", err.filename)

Checking `exists()` first and then opening has a hole in it: the file can disappear
between the two lines. Handling the exception is the version without the hole, and
`try`/`except` is module 09.

---

`exercises/` is next: `exercise_01.py` to `exercise_06.py`, `exercise_09.py`, and two
in `thinking.md` with nothing to run.

Module 09 takes the two loose ends this module left: what `with` actually is, and what
to do about the exception instead of checking first.